# YTsaurus Python Client

pip install ytsaurus-python-client

In [ ]:
from ytsaurus_python_client import DOYTHook, chyt_df, chyt_to_yt, chyt_raw
from ytsaurus_python_client import chyt_df_cli, chyt_to_yt_cli, _run_yt_cli, chyt_raw_cli


YT_PROXY="YT_PROXY"
YT_MY_DIR="YT_MY_DIR"


hook: DOYTHook = DOYTHook(yt_proxy=YT_PROXY, yt_query_result_temp_dir=YT_MY_DIR)

## YQL

### Пример для запроса

In [ ]:
query = """
-- Генерим таблицу
$rows = ListMap(ListFromRange(0, 120008), ($x) -> {
    return AsStruct(("item_" || cast($x as Utf8)) as id);
});
select id from as_table($rows) limit 25000
;
"""

### 1. hook.yql

#### Выполняет YQL-запрос и читает результат (только 10к строк)

In [ ]:
df = hook.yql(query)

In [5]:
df.shape # получили меньше строк, хотя limit 25000

(10000, 1)

In [6]:
df.head()

,id
0,item_0
1,item_1
2,item_2
3,item_3
4,item_4


#### wait=True, read_result=False - Выполняет YQL-запрос и ждёт его завершения, но не читает результат

In [ ]:
query_id = hook.yql("""
        INSERT INTO `my_table`
        SELECT
            *
        FROM `temp_table_zogJ38mY9l`
        limit 1000
        ;
""",  wait=True, read_result=False)

#### wait=False - Выполняет YQL-запрос и не ждет его завершения

In [ ]:
query_id = hook.yql("""
        INSERT INTO `my_table`
        SELECT
            *
        FROM `temp_table_zogJ38mY9l`
        limit 1000
        ;
""",  wait=False)

### 2. hook.yql_unlim - Чтение, всех строк

Запрос создаст временную таблицу (по-дефолту срок жизни 1 день, можно менять в аргументах, пример ниже)

In [ ]:
df = hook.yql_unlim(query)

In [10]:
df.shape # получили все строки

(25000, 1)

In [11]:
df.head()

,id
0,item_0
1,item_1
2,item_2
3,item_3
4,item_4


Можем задать более подробно Аргументы

In [ ]:
df = hook.yql_unlim(
    query, 
    temp_table_path = 'my_table_name123123', # можно не указывать, создас сам имя временой папки, которая по дефолту удалится чреез 1 день
    temp_table_expiration = '1d', # по-умолчаняю временная папка удалиться через 1 день
    overwrite = True # данные не перезаписываем, а просто добавляем новые
)

In [13]:
df.shape

(25000, 1)

In [14]:
df.head()

,id
0,item_0
1,item_1
2,item_2
3,item_3
4,item_4


### 3.1 hook.generate_yt_schema - Загрузка DataFrame в YT (небольшой датафрейм)

In [15]:
schema = hook.generate_yt_schema(df)
schema

[{'name': 'id', 'type': 'string'}]

In [ ]:
hook.upload_df_to_yt(df, yt_path=f"{YT_MY_DIR}/my_table_name", schema=schema, overwrite=True, log_result=True) # помните про overwrite, чтобы не удалить лишнее

In [18]:
# Иногда данные может распознать не правильно, для этого поправляем в ручном режиме типы данных
schema = hook.generate_yt_schema(df, custom_type_map={
    "id": "string"
})

# создаем таблицу и передаем схему
hook.client.create(
    "table",
    f"{YT_MY_DIR}/my_table_name_exmaple",
    attributes={"schema": schema},
)
schema


[{'name': 'id', 'type': 'string'}]

In [ ]:
hook.upload_df_to_yt(df, yt_path=f"{YT_MY_DIR}/my_table_name_exmaple", schema=schema, overwrite=True, log_result=True) # помните про overwrite, чтобы не удалить лишнее

In [ ]:
query = f"""
PRAGMA AutoCommit;
PRAGMA yt.DefaultOperationWeight = "100";

$path_table = '{YT_MY_DIR}/my_table_name_exmaple'
;

select * from $path_table limit 25000
;
"""

df = hook.yql_unlim(query) # можем использовать hook.yql(query), если не нужно вернуть все строки

In [23]:
df.shape

(25000, 1)

In [24]:
df.head()

,id
0,item_0
1,item_1
2,item_2
3,item_3
4,item_4


### 3.2 Загрузка DataFrame в YT (большой датафрейм), пример

In [25]:
from tqdm import tqdm

In [26]:
chunk_size = 10_000 # например по 10к, можем увеличить если нужно

total_rows = len(df)
total_chunks = (total_rows + chunk_size - 1) // chunk_size 

In [27]:
out_path = f"{YT_MY_DIR}/my_table_name_exmaple_chunks"

# подготовим схему (при необходимости добавь кастомный маппинг типов как выше был)
schema = hook.generate_yt_schema(df)

# если таблица уже есть и ты хочешь чистый перезапуск — снеси
if hook.exists(out_path):
    hook.client.remove(out_path, force=True)

# создаём пустую таблицу с нужной схемой ОДИН раз
hook.client.create("table", out_path, attributes={"schema": schema})

'5b886-24d3c1-60191-21131abd'

In [28]:
with tqdm(total=total_chunks, desc="Грузим данные по чанкам:", unit="chunk") as pbar:
    for i in range(0, total_rows, chunk_size):
        chunk_df = df.iloc[i:i + chunk_size]
        hook.upload_df_to_yt(
            chunk_df,                     # ← ВАЖНО: chunk_df, не df
            yt_path=out_path,
            schema=schema,
            overwrite=False,              # ← аппенд
            log_result=False
        )
        pbar.set_postfix({"rows": f"{i}–{min(i + chunk_size, total_rows)}"})
        pbar.update(1)

Грузим данные по чанкам:: 100%|████████████████████████████████████████████████████████████████████████████████| 3/3 [00:06<00:00,  2.13s/chunk, rows=20000–25000]


In [ ]:
query = f"""
PRAGMA AutoCommit;
PRAGMA yt.DefaultOperationWeight = "100";

$path_table = '{YT_MY_DIR}/my_table_name_exmaple_chunks'
;

select * from $path_table limit 25000
;
"""

df = hook.yql_unlim(query)

In [30]:
df.shape

(25000, 1)

In [31]:
df.head()

,id
0,item_0
1,item_1
2,item_2
3,item_3
4,item_4


## CHYT

### http

#### .chyt_df - Чтение данных в DataFrame

In [32]:
out_path = f"{YT_MY_DIR}/my_table_name_exmaple_chunks" # путь таблицы из предыдущих шагов

In [40]:
df = chyt_df(f"""
    select
    	*
    from '{out_path}'
    limit 10
""")

In [41]:
df.shape

(10, 2)

In [42]:
df.head()

,id,index
0,item_0,0
1,item_1,1
2,item_10,10
3,item_100,100
4,item_1000,1000


#### .chyt_df - Загрузка DataFrame

In [ ]:
out_path = f"{YT_MY_DIR}/my_table_chunks_chyt"

out_path

In [44]:
df = df.sort_values('id', kind='mergesort').reset_index() # сортируем тот столбец который передами, если таблица создается в первый раз
df.head()

,level_0,id,index
0,0,item_0,0
1,1,item_1,1
2,2,item_10,10
3,3,item_100,100
4,4,item_1000,1000


In [45]:
chyt_to_yt(df, out_path, overwrite=True, order_by=["id"])

In [46]:
df = chyt_df_cli(f"""select * from '{out_path}' limit 15""")

In [47]:
df.head()

,id,index
0,item_0,0
1,item_1,1
2,item_10,10
3,item_100,100
4,item_1000,1000


#### .chyt_raw - Выполнение сырого запроса CHYT

In [ ]:
txt = chyt_raw("""EXPLAIN SELECT 1""") # можно и другие, insert, create и тд
print(txt)

In [49]:
txt = chyt_raw(f"""describe table '{out_path}';""")
print(txt)

id	String					
index	Int64					



### cli

#### .chyt_df_cli - Чтение данных в DataFrame

In [50]:
out_path = f"{YT_MY_DIR}/my_table_name_exmaple_chunks" # путь таблицы из предыдущих шагов

In [51]:
df = chyt_df_cli(f"""
    select
    	*
    from '{out_path}'
    limit 10
""")

In [52]:
df.shape

(10, 1)

In [53]:
df.head()

,id
0,item_0
1,item_1
2,item_2
3,item_3
4,item_4


#### .chyt_df_cli - Загрузка DataFrame

In [ ]:
out_path = f"{YT_MY_DIR}/my_table_chunks_chyt_cli_t"

out_path

In [56]:
df = df.sort_values('id', kind='mergesort').reset_index() # сортируем тот столбец который передами, если таблица создается в первый раз
df.head()

,index,id
0,0,item_0
1,1,item_1
2,2,item_2
3,3,item_3
4,4,item_4


In [58]:
chyt_to_yt_cli(df, out_path, overwrite=True, order_by=["id"])

In [ ]:
# А можно и так

# YT_BINARY = "yt"

# txt = _run_yt_cli([
#     YT_BINARY,
#     "remove",
#     out_path,
#     "--force",
# ])

# print(txt) 

In [59]:
df = chyt_df_cli(f"""select * from '{out_path}' limit 15""")

In [60]:
df.head()

,id,index
0,item_0,0
1,item_1,1
2,item_2,2
3,item_3,3
4,item_4,4


#### .chyt_raw_cli - Выполнение сырого запроса CHYT

In [ ]:
txt = chyt_raw_cli("""EXPLAIN SELECT 1""") # можно и другие, insert, create и тд
print(txt)

In [62]:
txt = chyt_raw(f"""describe table '{out_path}';""")
print(txt)

id	String					
index	Int64					



In [63]:
txt = chyt_raw_cli(f"""
DROP TABLE IF EXISTS `{out_path}`
""")

print(txt)

In [6]:
# txt = chyt_raw(f"""describe table '{out_path}';""")
# print(txt)

#### Как сделать insert

In [ ]:
out_path = "chyt_insert_example"

chyt_raw_cli(f"""
DROP TABLE IF EXISTS `{out_path}`
""")

chyt_raw_cli(f"""
CREATE TABLE `{out_path}`
(
    `id` Int64,
    `name` String,
    `amount` Double
)
ENGINE = YtTable()
ORDER BY (`id`)
""")

''

In [4]:
chyt_raw_cli(f"""
INSERT INTO `{out_path}`
SELECT
    1 AS id,
    'test_1' AS name,
    100.5 AS amount
UNION ALL
SELECT
    2 AS id,
    'test_2' AS name,
    200.0 AS amount
""")

''

In [5]:
df = chyt_df_cli(f"""
SELECT *
FROM `{out_path}`
ORDER BY id
""")

df

,id,name,amount
0,1,test_1,100.5
1,2,test_2,200.0


## Дополнительные функции из библиотеки

In [ ]:
# Получить список файлов / директорий
hook.ls(f"{YT_MY_DIR}")

In [66]:
# Удаление таблицы
hook.client.remove(f"{YT_MY_DIR}/sample_table_from_py_temp", recursive=False, force=True)

b'{}'

In [67]:
# Проверить, существует ли таблица или папка
hook.exists(f"{YT_MY_DIR}/sample_table_from_py_temp")

false

In [ ]:
# Получить схему таблицы
hook.client.get("table/1d/2025-04-23/@schema", format="json")

In [ ]:
# Все атрибуты
hook.client.get("table/1d/2025-04-23/@")  

In [ ]:
# Кол-во строк
hook.client.get("table/1d/2025-04-23/@row_count")

22626273792